
# Amazon Reviews — Performance e Testes

## Objetivo

Avaliar a performance, a consistência e a confiabilidade
das transformações do projeto Amazon Reviews Big Data
Analytics, utilizando Apache Spark e Databricks Serverless.

## Escopo

- Análise de planos de execução.
- Investigação de operações custosas.
- Comparação de estratégias de consulta.
- Validação de qualidade entre as camadas Medallion.
- Verificação de propriedades de idempotência.
- Registro de resultados e oportunidades de otimização.

## Ambiente

- Plataforma: Databricks Free Edition
- Compute: Serverless
- Linguagem: Python / PySpark
- Catálogo: workspace
- Schema: amazon_reviews
- Formato das tabelas: Delta Lake

## Boas práticas

- Priorizar consultas de leitura.
- Evitar processamento e recomputação desnecessários.
- Não modificar as tabelas de produção.
- Não executar novamente o Lakeflow sem necessidade.
- Não utilizar collect() para grandes volumes de dados.
- Registrar resultados reais, sem estimar tempos.


In [0]:

resultado = spark.sql("SELECT 1 AS ambiente_ok")

resultado.show()


In [0]:

# Leitura da tabela Gold
df_produtos = spark.table(
    "workspace.amazon_reviews.gold_genie_product_analytics"
)

# Seleção das colunas necessárias e aplicação de filtro
df_consulta = (
    df_produtos
    .select("parent_asin", "reviews_count")
    .filter("reviews_count >= 100")
)

# Exibir o plano físico de execução
df_consulta.explain(mode="formatted")


In [0]:

total_produtos_100 = df_consulta.count()

print(
    "Produtos com 100 ou mais avaliações:",
    total_produtos_100
)


In [0]:

from pyspark.sql import functions as F

# Ler a tabela Gold
df_produtos = spark.table(
    "workspace.amazon_reviews.gold_genie_product_analytics"
)

# Agrupar produtos por categoria
df_agregacao = (
    df_produtos
    .groupBy("main_category")
    .agg(
        F.count("*").alias("total_produtos"),
        F.sum("reviews_count").alias("total_avaliacoes")
    )
)

# Inspecionar o plano físico
df_agregacao.explain(mode="formatted")


In [0]:

df_agregacao.show(n=20, truncate=False)


In [0]:

# Reconciliação Bronze x Silver

df_reconciliacao = spark.sql("""
    WITH bronze AS (
        SELECT COUNT(*) AS total_bronze
        FROM workspace.amazon_reviews.bronze_reviews_raw
    ),
    silver AS (
        SELECT COUNT(*) AS total_silver
        FROM workspace.amazon_reviews.silver_reviews_clean
    )

    SELECT
        b.total_bronze,
        s.total_silver,

        b.total_bronze - s.total_silver
            AS registros_removidos,

        ROUND(
            100.0 * (b.total_bronze - s.total_silver)
            / NULLIF(b.total_bronze, 0),
            2
        ) AS percentual_removido

    FROM bronze b
    CROSS JOIN silver s
""")

df_reconciliacao.show(truncate=False)


In [0]:

from pyspark.sql import functions as F

# Campos originais das avaliações
colunas_origem = [
    "rating",
    "title",
    "text",
    "images",
    "asin",
    "parent_asin",
    "user_id",
    "timestamp",
    "helpful_vote",
    "verified_purchase"
]

# Leitura da Bronze
df_bronze = spark.table(
    "workspace.amazon_reviews.bronze_reviews_raw"
)

# Verificar se o schema possui todos os campos
colunas_ausentes = [
    coluna for coluna in colunas_origem
    if coluna not in df_bronze.columns
]

if colunas_ausentes:
    raise ValueError(
        f"Colunas ausentes: {colunas_ausentes}"
    )

# Criar uma assinatura para cada avaliação
df_assinaturas = df_bronze.select(
    F.sha2(
        F.to_json(
            F.struct(
                *[F.col(c) for c in colunas_origem]
            )
        ),
        256
    ).alias("assinatura")
)

# Agrupar avaliações com a mesma assinatura
df_grupos = (
    df_assinaturas
    .groupBy("assinatura")
    .agg(F.count("*").alias("ocorrencias"))
)

# Consolidar indicadores de duplicação
df_duplicatas = df_grupos.agg(
    F.sum(
        F.when(F.col("ocorrencias") > 1, 1)
         .otherwise(0)
    ).alias("grupos_duplicados"),

    F.sum(
        F.when(
            F.col("ocorrencias") > 1,
            F.col("ocorrencias")
        ).otherwise(0)
    ).alias("registros_em_grupos_duplicados"),

    F.sum(
        F.col("ocorrencias") - 1
    ).alias("copias_excedentes"),

    F.max("ocorrencias").alias("max_repeticoes")
)

df_duplicatas.show(truncate=False)


In [0]:

# Integridade Bronze x Silver

# As dez colunas originais das avaliações
colunas_origem = [
    "rating",
    "title",
    "text",
    "images",
    "asin",
    "parent_asin",
    "user_id",
    "timestamp",
    "helpful_vote",
    "verified_purchase"
]

# 1. Reconstruir os registros únicos da Bronze
df_bronze_unica = (
    spark.table(
        "workspace.amazon_reviews.bronze_reviews_raw"
    )
    .select(*colunas_origem)
    .dropDuplicates()
)

# 2. Selecionar os mesmos campos da Silver
df_silver_origem = (
    spark.table(
        "workspace.amazon_reviews.silver_reviews_clean"
    )
    .select(*colunas_origem)
)

# 3. Registros esperados que faltam na Silver
registros_ausentes = (
    df_bronze_unica
    .exceptAll(df_silver_origem)
    .count()
)

# 4. Registros inesperados na Silver
registros_inesperados = (
    df_silver_origem
    .exceptAll(df_bronze_unica)
    .count()
)

# 5. Exibir os resultados
print("=== INTEGRIDADE BRONZE X SILVER ===")
print("Registros ausentes na Silver:", registros_ausentes)
print("Registros inesperados na Silver:", registros_inesperados)

if registros_ausentes == 0 and registros_inesperados == 0:
    print("RESULTADO: APROVADO")
else:
    print("RESULTADO: INVESTIGAR DIVERGENCIAS")


In [0]:

# Reconciliação Silver x Gold

df_reconciliacao_gold = spark.sql("""
    WITH silver_reviews AS (
        SELECT
            COUNT(*) AS avaliacoes_silver,
            COUNT(DISTINCT parent_asin) AS produtos_avaliados_silver
        FROM workspace.amazon_reviews.silver_reviews_clean
    ),

    silver_products AS (
        SELECT
            COUNT(*) AS produtos_silver
        FROM workspace.amazon_reviews.silver_products_clean
    ),

    gold AS (
        SELECT
            COUNT(*) AS produtos_gold,
            COALESCE(SUM(reviews_count), 0)
                AS avaliacoes_gold,

            SUM(
                CASE WHEN reviews_count > 0 THEN 1 ELSE 0 END
            ) AS produtos_avaliados_gold,

            SUM(
                CASE WHEN reviews_count = 0 THEN 1 ELSE 0 END
            ) AS produtos_sem_avaliacoes_gold

        FROM workspace.amazon_reviews.gold_genie_product_analytics
    )

    SELECT
        sr.avaliacoes_silver,
        g.avaliacoes_gold,

        sp.produtos_silver,
        g.produtos_gold,

        sr.produtos_avaliados_silver,
        g.produtos_avaliados_gold,
        g.produtos_sem_avaliacoes_gold,

        CASE
            WHEN sr.avaliacoes_silver = g.avaliacoes_gold
             AND sp.produtos_silver = g.produtos_gold
             AND sr.produtos_avaliados_silver =
                 g.produtos_avaliados_gold
             AND g.produtos_gold =
                 g.produtos_avaliados_gold +
                 g.produtos_sem_avaliacoes_gold
            THEN 'APROVADO'
            ELSE 'INVESTIGAR'
        END AS status_reconciliacao

    FROM silver_reviews sr
    CROSS JOIN silver_products sp
    CROSS JOIN gold g
""")

df_reconciliacao_gold.show(truncate=False)


In [0]:

#  Validação de avaliações por produto

df_validacao_produtos = spark.sql("""
    WITH silver_por_produto AS (
        SELECT
            parent_asin,
            COUNT(*) AS avaliacoes_silver
        FROM workspace.amazon_reviews.silver_reviews_clean
        GROUP BY parent_asin
    ),

    comparacao AS (
        SELECT
            COALESCE(g.parent_asin, s.parent_asin)
                AS parent_asin,

            CASE
                WHEN g.parent_asin IS NULL THEN 1
                WHEN NOT (
                    g.reviews_count <=>
                    COALESCE(s.avaliacoes_silver, 0)
                ) THEN 1
                ELSE 0
            END AS divergente,

            CASE
                WHEN g.parent_asin IS NULL THEN 1
                ELSE 0
            END AS ausente_na_gold,

            CASE
                WHEN s.parent_asin IS NULL THEN 1
                ELSE 0
            END AS sem_avaliacoes_na_silver

        FROM workspace.amazon_reviews.gold_genie_product_analytics g
        FULL OUTER JOIN silver_por_produto s
            ON g.parent_asin = s.parent_asin
    )

    SELECT
        COUNT(*) AS produtos_comparados,
        SUM(divergente) AS produtos_divergentes,
        SUM(ausente_na_gold) AS produtos_ausentes_gold,
        SUM(sem_avaliacoes_na_silver)
            AS produtos_sem_avaliacoes

    FROM comparacao
""")

df_validacao_produtos.show(truncate=False)


In [0]:

#  Unicidade e integridade do parent_asin

df_validacao_chaves = spark.sql("""
    WITH ocorrencias AS (
        SELECT
            parent_asin,
            COUNT(*) AS quantidade
        FROM workspace.amazon_reviews.gold_genie_product_analytics
        GROUP BY parent_asin
    )

    SELECT
        SUM(quantidade) AS total_registros,

        COUNT(*) AS chaves_distintas,

        SUM(
            CASE
                WHEN quantidade > 1 THEN 1
                ELSE 0
            END
        ) AS chaves_duplicadas,

        SUM(quantidade - 1)
            AS registros_excedentes,

        SUM(
            CASE
                WHEN parent_asin IS NULL THEN quantidade
                ELSE 0
            END
        ) AS ids_nulos,

        SUM(
            CASE
                WHEN TRIM(parent_asin) = '' THEN quantidade
                ELSE 0
            END
        ) AS ids_vazios

    FROM ocorrencias
""")

df_validacao_chaves.show(truncate=False)


In [0]:

#  Validar avaliações verificadas e votos úteis

df_validacao_metricas = spark.sql("""
    WITH silver_por_produto AS (
        SELECT
            parent_asin,

            COUNT_IF(verified_purchase = true)
                AS verificadas_silver,

            SUM(COALESCE(helpful_vote, 0))
                AS votos_silver

        FROM workspace.amazon_reviews.silver_reviews_clean
        GROUP BY parent_asin
    ),

    comparacao AS (
        SELECT
            g.parent_asin,

            g.verified_review_count AS verificadas_gold,
            COALESCE(s.verificadas_silver, 0)
                AS verificadas_silver,

            g.total_helpful_votes AS votos_gold,
            COALESCE(s.votos_silver, 0)
                AS votos_silver

        FROM workspace.amazon_reviews.gold_genie_product_analytics g
        LEFT JOIN silver_por_produto s
            ON g.parent_asin = s.parent_asin
    )

    SELECT
        COUNT(*) AS produtos_comparados,

        SUM(CASE
            WHEN NOT (verificadas_gold <=> verificadas_silver)
            THEN 1 ELSE 0
        END) AS divergencias_verificadas,

        SUM(CASE
            WHEN NOT (votos_gold <=> votos_silver)
            THEN 1 ELSE 0
        END) AS divergencias_votos,

        SUM(verificadas_silver) AS total_verificadas_silver,
        SUM(verificadas_gold) AS total_verificadas_gold,

        SUM(votos_silver) AS total_votos_silver,
        SUM(votos_gold) AS total_votos_gold

    FROM comparacao
""")

df_validacao_metricas.show(truncate=False)


In [0]:

#  Validar médias e percentuais da Gold

df_validacao_medias = spark.sql("""
    WITH silver_por_produto AS (
        SELECT
            parent_asin,

            AVG(CAST(rating AS DOUBLE))
                AS media_silver,

            100.0 * COUNT_IF(verified_purchase = true)
                / NULLIF(COUNT(*), 0)
                AS percentual_silver

        FROM workspace.amazon_reviews.silver_reviews_clean
        GROUP BY parent_asin
    ),

    comparacao AS (
        SELECT
            g.parent_asin,

            CASE
                WHEN g.average_review_rating IS NULL
                 AND s.media_silver IS NULL
                THEN 0

                WHEN g.average_review_rating IS NOT NULL
                 AND s.media_silver IS NOT NULL
                 AND ABS(
                     CAST(g.average_review_rating AS DOUBLE)
                     - s.media_silver
                 ) <= 0.011
                THEN 0

                ELSE 1
            END AS divergencia_media,

            CASE
                WHEN g.verified_review_pct IS NULL
                 AND s.percentual_silver IS NULL
                THEN 0

                WHEN g.verified_review_pct IS NOT NULL
                 AND s.percentual_silver IS NOT NULL
                 AND ABS(
                     CAST(g.verified_review_pct AS DOUBLE)
                     - s.percentual_silver
                 ) <= 0.011
                THEN 0

                ELSE 1
            END AS divergencia_percentual,

            CASE
                WHEN g.reviews_count = 0
                 AND g.average_review_rating IS NULL
                 AND g.verified_review_pct IS NULL
                THEN 1
                ELSE 0
            END AS produto_sem_reviews_valido

        FROM workspace.amazon_reviews.gold_genie_product_analytics g

        LEFT JOIN silver_por_produto s
            ON g.parent_asin = s.parent_asin
    )

    SELECT
        COUNT(*) AS produtos_comparados,

        SUM(divergencia_media)
            AS divergencias_media,

        SUM(divergencia_percentual)
            AS divergencias_percentual,

        SUM(produto_sem_reviews_valido)
            AS produtos_sem_reviews_validos

    FROM comparacao
""")

df_validacao_medias.show(truncate=False)


In [0]:

#  Reconciliação mensal Silver x Gold

# Utilizar o mesmo fuso da transformação Gold
spark.conf.set("spark.sql.session.timeZone", "UTC")

df_validacao_mensal = spark.sql("""
    WITH silver_mensal AS (
        SELECT
            CAST(
                DATE_TRUNC('MONTH', review_timestamp_utc)
                AS DATE
            ) AS review_month,

            COUNT(*) AS avaliacoes_silver,

            COUNT_IF(verified_purchase = true)
                AS verificadas_silver

        FROM workspace.amazon_reviews.silver_reviews_clean

        GROUP BY
            CAST(
                DATE_TRUNC('MONTH', review_timestamp_utc)
                AS DATE
            )
    ),

    comparacao AS (
        SELECT
            COALESCE(s.review_month, g.review_month)
                AS review_month,

            CASE
                WHEN g.review_month IS NULL
                THEN 1 ELSE 0
            END AS ausente_gold,

            CASE
                WHEN s.review_month IS NULL
                THEN 1 ELSE 0
            END AS extra_gold,

            CASE
                WHEN NOT (
                    s.avaliacoes_silver <=> g.reviews_count
                )
                THEN 1 ELSE 0
            END AS divergencia_avaliacoes,

            CASE
                WHEN NOT (
                    s.verificadas_silver
                    <=> g.verified_review_count
                )
                THEN 1 ELSE 0
            END AS divergencia_verificadas,

            g.reviews_count,
            g.verified_review_count

        FROM silver_mensal s

        FULL OUTER JOIN
            workspace.amazon_reviews.gold_monthly_review_metrics g

        ON s.review_month = g.review_month
    )

    SELECT
        COUNT(*) AS meses_comparados,

        SUM(ausente_gold) AS meses_ausentes_gold,

        SUM(extra_gold) AS meses_extras_gold,

        SUM(divergencia_avaliacoes)
            AS divergencias_avaliacoes,

        SUM(divergencia_verificadas)
            AS divergencias_verificadas,

        SUM(reviews_count)
            AS total_avaliacoes_gold,

        SUM(verified_review_count)
            AS total_verificadas_gold

    FROM comparacao
""")

df_validacao_mensal.show(truncate=False)


In [0]:

#  Validação das métricas derivadas mensais

# Manter o mesmo fuso utilizado na Gold mensal
spark.conf.set("spark.sql.session.timeZone", "UTC")

df_validacao_mensal_derivada = spark.sql("""
    WITH silver_mensal AS (
        SELECT
            CAST(
                DATE_TRUNC('MONTH', review_timestamp_utc)
                AS DATE
            ) AS review_month,

            AVG(CAST(rating AS DOUBLE))
                AS media_silver,

            100.0 * COUNT_IF(verified_purchase = true)
                / NULLIF(COUNT(*), 0)
                AS percentual_silver,

            COUNT(DISTINCT parent_asin)
                AS produtos_distintos_silver

        FROM workspace.amazon_reviews.silver_reviews_clean

        GROUP BY
            CAST(
                DATE_TRUNC('MONTH', review_timestamp_utc)
                AS DATE
            )
    ),

    comparacao AS (
        SELECT
            g.review_month,

            CASE
                WHEN g.average_review_rating IS NULL
                 AND s.media_silver IS NULL
                THEN 0

                WHEN g.average_review_rating IS NOT NULL
                 AND s.media_silver IS NOT NULL
                 AND ABS(
                    CAST(g.average_review_rating AS DOUBLE)
                    - s.media_silver
                 ) <= 0.011
                THEN 0

                ELSE 1
            END AS divergencia_media,

            CASE
                WHEN g.verified_review_pct IS NULL
                 AND s.percentual_silver IS NULL
                THEN 0

                WHEN g.verified_review_pct IS NOT NULL
                 AND s.percentual_silver IS NOT NULL
                 AND ABS(
                    CAST(g.verified_review_pct AS DOUBLE)
                    - s.percentual_silver
                 ) <= 0.011
                THEN 0

                ELSE 1
            END AS divergencia_percentual,

            CASE
                WHEN g.reviewed_products_count
                     <=> s.produtos_distintos_silver
                THEN 0
                ELSE 1
            END AS divergencia_produtos

        FROM workspace.amazon_reviews.gold_monthly_review_metrics g

        LEFT JOIN silver_mensal s
            ON g.review_month = s.review_month
    )

    SELECT
        COUNT(*) AS meses_comparados,

        SUM(divergencia_media)
            AS divergencias_media,

        SUM(divergencia_percentual)
            AS divergencias_percentual,

        SUM(divergencia_produtos)
            AS divergencias_produtos

    FROM comparacao
""")

df_validacao_mensal_derivada.show(truncate=False)


In [0]:

#  Baseline das sete tabelas Delta

df_baseline = spark.sql("""
    SELECT
        'Bronze' AS camada,
        'bronze_reviews_raw' AS tabela,
        COUNT(*) AS total_registros
    FROM workspace.amazon_reviews.bronze_reviews_raw

    UNION ALL

    SELECT
        'Bronze',
        'bronze_products_raw',
        COUNT(*)
    FROM workspace.amazon_reviews.bronze_products_raw

    UNION ALL

    SELECT
        'Silver',
        'silver_reviews_clean',
        COUNT(*)
    FROM workspace.amazon_reviews.silver_reviews_clean

    UNION ALL

    SELECT
        'Silver',
        'silver_products_clean',
        COUNT(*)
    FROM workspace.amazon_reviews.silver_products_clean

    UNION ALL

    SELECT
        'Gold',
        'gold_product_metrics',
        COUNT(*)
    FROM workspace.amazon_reviews.gold_product_metrics

    UNION ALL

    SELECT
        'Gold',
        'gold_monthly_review_metrics',
        COUNT(*)
    FROM workspace.amazon_reviews.gold_monthly_review_metrics

    UNION ALL

    SELECT
        'Gold',
        'gold_genie_product_analytics',
        COUNT(*)
    FROM workspace.amazon_reviews.gold_genie_product_analytics

    ORDER BY camada, tabela
""")

print("=== BASELINE DE IDEMPOTENCIA ===")
df_baseline.show(truncate=False)


In [0]:

#  Idempotência após Refresh
# Comparação das sete tabelas com a baseline

df_idempotencia = spark.sql("""
    WITH baseline AS (
        SELECT * FROM VALUES
            ('Bronze', 'bronze_reviews_raw', 701528),
            ('Bronze', 'bronze_products_raw', 112590),
            ('Silver', 'silver_reviews_clean', 694253),
            ('Silver', 'silver_products_clean', 112590),
            ('Gold', 'gold_product_metrics', 112565),
            ('Gold', 'gold_monthly_review_metrics', 265),
            ('Gold', 'gold_genie_product_analytics', 112590)
        AS t(camada, tabela, total_antes)
    ),

    atual AS (
        SELECT
            'bronze_reviews_raw' AS tabela,
            COUNT(*) AS total_depois
        FROM workspace.amazon_reviews.bronze_reviews_raw

        UNION ALL

        SELECT 'bronze_products_raw', COUNT(*)
        FROM workspace.amazon_reviews.bronze_products_raw

        UNION ALL

        SELECT 'silver_reviews_clean', COUNT(*)
        FROM workspace.amazon_reviews.silver_reviews_clean

        UNION ALL

        SELECT 'silver_products_clean', COUNT(*)
        FROM workspace.amazon_reviews.silver_products_clean

        UNION ALL

        SELECT 'gold_product_metrics', COUNT(*)
        FROM workspace.amazon_reviews.gold_product_metrics

        UNION ALL

        SELECT 'gold_monthly_review_metrics', COUNT(*)
        FROM workspace.amazon_reviews.gold_monthly_review_metrics

        UNION ALL

        SELECT 'gold_genie_product_analytics', COUNT(*)
        FROM workspace.amazon_reviews.gold_genie_product_analytics
    )

    SELECT
        b.camada,
        b.tabela,
        b.total_antes,
        a.total_depois,

        a.total_depois - b.total_antes
            AS diferenca,

        CASE
            WHEN a.total_depois = b.total_antes
            THEN 'APROVADO'
            ELSE 'INVESTIGAR'
        END AS status

    FROM baseline b

    INNER JOIN atual a
        ON b.tabela = a.tabela

    ORDER BY b.camada, b.tabela
""")

df_idempotencia.show(n=20, truncate=False)


In [0]:

#  Consistência de métricas após Refresh

df_metricas_pos_refresh = spark.sql("""
    WITH genie AS (
        SELECT
            COUNT(*) AS produtos,

            SUM(
                CASE WHEN reviews_count > 0
                THEN 1 ELSE 0 END
            ) AS produtos_avaliados,

            SUM(
                CASE WHEN reviews_count = 0
                THEN 1 ELSE 0 END
            ) AS produtos_sem_avaliacoes,

            SUM(reviews_count) AS avaliacoes,

            SUM(verified_review_count)
                AS verificadas,

            SUM(total_helpful_votes)
                AS votos_uteis

        FROM workspace.amazon_reviews.gold_genie_product_analytics
    ),

    mensal AS (
        SELECT
            COUNT(*) AS meses,

            SUM(reviews_count)
                AS avaliacoes_mensais,

            SUM(verified_review_count)
                AS verificadas_mensais

        FROM workspace.amazon_reviews.gold_monthly_review_metrics
    )

    SELECT
        g.produtos,
        g.produtos_avaliados,
        g.produtos_sem_avaliacoes,
        g.avaliacoes,
        g.verificadas,
        g.votos_uteis,
        m.meses,

        CASE
            WHEN g.avaliacoes = m.avaliacoes_mensais
             AND g.verificadas = m.verificadas_mensais
            THEN 'CONSISTENTE'
            ELSE 'DIVERGENTE'
        END AS consistencia_gold

    FROM genie g
    CROSS JOIN mensal m
""")

df_metricas_pos_refresh.show(truncate=False)
